# KIS — pilot OCR độc lập, CPU
Bật **Internet**, không cần GPU. Thêm ZIP pilot làm **private Dataset**. Chỉ <=12 ảnh, không có caption/qrels/credentials. Notebook tạo venv riêng, không cài requirements Windows hoặc thay môi trường KIS. Đây là runtime/development pilot, chưa phải benchmark accuracy độc lập.


In [ ]:
from pathlib import Path
import hashlib, json, sys, zipfile, subprocess, venv
assert (3, 10) <= sys.version_info[:2] <= (3, 12), 'This pinned pilot requires Python 3.10-3.12; do not unpin NumPy to force it'
expected_manifest_sha = '68fcc552f1eaadb4a46d8aa453fb3f0111df1c3dc3b84b980a60dfc254b30a78'
code_files = ['kis/__init__.py','kis/paths.py','kis/artifact_io.py','kis/ocr.py','tools/__init__.py','tools/build_ocr.py']
inputs = list(Path('/kaggle/input').rglob('kis-ocr-pilot-20261104.zip'))
work = Path('/kaggle/working/kis-ocr-pilot')
assert not work.exists(), 'Use a fresh working directory/session; do not overwrite an earlier pilot'
work.mkdir()
root = work / 'code'
root.mkdir()
if inputs:
    assert len(inputs) == 1, 'Add only one pilot ZIP'
    with zipfile.ZipFile(inputs[0]) as z:
        assert len(z.namelist()) <= 200
        assert len(set(z.namelist())) == len(z.namelist()), 'Duplicate ZIP paths'
        assert sum(info.file_size for info in z.infolist()) < 100*1024*1024
        payload = z.read('manifest.json')
        assert hashlib.sha256(payload).hexdigest() == expected_manifest_sha, 'Wrong pilot manifest'
        m = json.loads(payload)
        assert set(m['source_sha256']) == set(code_files) and 1 <= len(m['frames']) <= 12
        allowed = {'manifest.json','requirements-ocr.txt',*code_files,*m['mapping_files']}
        allowed.update('images/'+frame['image_relative'] for frame in m['frames'])
        assert set(z.namelist()) == allowed, 'Unexpected pilot archive contents'
        for info in z.infolist():
            p = Path(info.filename)
            assert not p.is_absolute() and '..' not in p.parts and '\\' not in info.filename
            assert info.file_size < 50*1024*1024
        z.extractall(root)
else:
    # Kaggle can expand uploaded ZIPs; accept only the manifest's allowlist.
    import shutil
    manifests = [p for p in Path('/kaggle/input').rglob('manifest.json')
                 if (p.parent / 'tools/build_ocr.py').is_file()]
    assert len(manifests) == 1, 'Add the pilot ZIP or its expanded contents'
    base = manifests[0].parent
    manifest = json.loads(manifests[0].read_text())
    assert hashlib.sha256(manifests[0].read_bytes()).hexdigest() == expected_manifest_sha
    assert set(manifest['source_sha256']) == set(code_files) and 1 <= len(manifest['frames']) <= 12
    names = ['manifest.json', 'requirements-ocr.txt', *manifest['source_sha256'], *manifest['mapping_files']]
    names += ['images/' + frame['image_relative'] for frame in manifest['frames']]
    for name in dict.fromkeys(names):
        p = Path(name)
        assert not p.is_absolute() and '..' not in p.parts
        target = root / p
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(base / p, target)
manifest = json.loads((root / 'manifest.json').read_text())
assert hashlib.sha256((root/'manifest.json').read_bytes()).hexdigest() == expected_manifest_sha
assert hashlib.sha256((root/'requirements-ocr.txt').read_bytes()).hexdigest() == manifest['requirements_sha256']
for name, expected in manifest['source_sha256'].items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest() == expected, name
for frame in manifest['frames']:
    assert hashlib.sha256((root/'images'/frame['image_relative']).read_bytes()).hexdigest() == frame['image_sha256']
for name, expected in manifest['mapping_files'].items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest() == expected
print('Frames:', len(manifest['frames']), 'CPU-only; no labels/captions')


In [ ]:
env = work / 'venv'
venv.EnvBuilder(with_pip=True, system_site_packages=False).create(env)
python = env / 'bin/python'
subprocess.run([str(python), '-m', 'pip', 'install', '-q', '-r', str(root/'requirements-ocr.txt')], check=True)


In [ ]:
# Download the two pinned SMALL OCR checkpoints, not PaddleOCR-VL/PP-OCRv6 defaults.
setup = '''
import hashlib,json,importlib.metadata
from pathlib import Path
from huggingface_hub import snapshot_download
root=Path.cwd()
m=json.loads((root/'manifest.json').read_text())
models=[]
for name,repo,revision in m['models']:
    directory=Path(snapshot_download(repo_id=repo,revision=revision,
        allow_patterns=['config.json','inference.json','inference.pdiparams','inference.yml']))
    files={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in directory.iterdir() if p.is_file()}
    models.append(dict(name=name,repo=repo,revision=revision,directory=str(directory),files_sha256=files))
settings=dict(backend='paddleocr',package_version=importlib.metadata.version('paddleocr'),
              paddle_version=importlib.metadata.version('paddlepaddle'),models=models)
assert settings['package_version']=='3.7.0' and settings['paddle_version']=='3.3.1'
(root/'engine.local.json').write_text(json.dumps(settings,indent=2))
'''
subprocess.run([str(python), '-c', setup], cwd=root, check=True)


In [ ]:
# Bounded real inference; no generated captions and no automatic region guesses.
output = work / 'results'
output.mkdir()
command = [str(python), '-B', '-m', 'tools.build_ocr', 'extract',
           '--input', str(root/'manifest.json'), '--image-root', str(root/'images'),
           '--mapping-dir', str(root/'mapping'), '--engine', str(root/'engine.local.json'),
           '--output', str(output/'ocr_detections.json'), '--max-frames', '12']
completed = subprocess.run(command, cwd=root, check=False)
assert (output/'ocr_detections.json').is_file(), 'Model/runtime initialization failed; send the traceback'
results = json.loads((output/'ocr_detections.json').read_text())
assert completed.returncode in (0, 2)
assert (completed.returncode == 0) == results['complete']
if not results['complete']:
    print('INCOMPLETE PILOT: keep error records unknown; return the ZIP and logs, do not label them negative')
import shutil
shutil.copyfile(root/'manifest.json', output/'input_manifest.json')
packages = subprocess.check_output([str(python), '-m', 'pip', 'freeze'], text=True)
(output/'packages.txt').write_text(packages)
for name, expected in manifest['source_sha256'].items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest() == expected, 'Code changed during pilot'
(output/'run_context.json').write_text(json.dumps(dict(device='cpu', complete=results['complete'],
    source_sha256=manifest['source_sha256'], input_manifest_sha256=hashlib.sha256((root/'manifest.json').read_bytes()).hexdigest(),
    frames=len(results['frames']), verified=False), indent=2))
archive = shutil.make_archive(str(work/'ocr-pilot-results'), 'zip', output)
from IPython.display import FileLink, display
display(FileLink(archive))
